# Reformat and Cleaning

Notebook ini fokus ke tahap normalisasi data multi-daerah.

Tujuan utama:
- standarisasi schema
- bersihkan bad rows
- imputasi dengan konteks daerah
- export satu dataset bersih yang siap dipakai training

Catatan desain:
- tidak ada train/test split di notebook ini
- split final dilakukan di notebook training per `nama_daerah`


## 1. Define Library and Path

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

raw_path = Path('../data/csv/master_data_ocds.csv')
cleaned_root = Path('../data/cleaned')
csv_root = Path('../data/csv')

cleaned_data_path = cleaned_root / 'cleaned_data.csv'
normalized_data_path = csv_root / 'normalized_data.csv'
summary_path = cleaned_root / 'cleaned_data_summary_by_daerah.csv'

# name reformatting (better reformatting)
REGION_NAME_OVERRIDES = {
    'Dki Jakarta': 'DKI Jakarta',
    'Daerah Istimewa Yogyakarta': 'Daerah Istimewa Yogyakarta',
}

def standardize_region_name(value):
    cleaned = str(value or '').strip().title()
    return REGION_NAME_OVERRIDES.get(cleaned, cleaned)

cleaned_root.mkdir(parents=True, exist_ok=True)
csv_root.mkdir(parents=True, exist_ok=True)


## 2. Load Raw Data

In [2]:
data = pd.read_csv(raw_path)

print(f'Loaded raw data: {len(data):,} rows, {len(data.columns)} columns')
print('\nInitial columns:')
print(data.columns.tolist())
print('\nInitial missing values:')
print(data.isnull().sum().to_string())

display(data.head(3))


Loaded raw data: 145,922 rows, 20 columns

Initial columns:
['lspe_id', 'nama_daerah', 'source_year', 'source_file', 'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainProcurementCategory', 'tender_minValue', 'tender_datePublished', 'tender_status', 'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier']

Initial missing values:
lspe_id                        0
nama_daerah                    0
source_year                    0
source_file                    0
ocid                           0
release_id                     0
date                           0
buyer_name                 18650
buyer_id                   16768
tender_id                      0
tender_title                   0
mainProcurementCategory        0
tender_minValue              151
tender_datePublished           0
tender_status                  0
award_id                       0
award_title                    0
award_date                     0
award_value       

,lspe_id,nama_daerah,source_year,source_file,ocid,release_id,date,buyer_name,buyer_id,tender_id,tender_title,mainProcurementCategory,tender_minValue,tender_datePublished,tender_status,award_id,award_title,award_date,award_value,award_supplier
0,106,Aceh,2015,ocds-data-tender-2015-106.json,ocds-20h3g7-8600106,ocds-20h3g7-8600106-1,2015-03-09T00:00:00.000000Z,Pemerintah Daerah Provinsi Aceh,D1,8600106,Pembangunan Gedung Kantor PT. BANK ACEH Cabang...,works,1.335736e+10,2015-02-09T00:00:00.000000Z,complete,8600106-1,Pembangunan Gedung Kantor PT. BANK ACEH Cabang...,2015-03-09T00:00:00.000000Z,1.187118e+10,PT RES KARYA
1,106,Aceh,2015,ocds-data-tender-2015-106.json,ocds-20h3g7-8604106,ocds-20h3g7-8604106-1,2015-03-20T00:00:00.000000Z,NaN,NaN,8604106,Pengadaan Jasa Cleaning Servis,services,2.291846e+09,2015-03-06T00:00:00.000000Z,complete,8604106-1,Pengadaan Jasa Cleaning Servis,2015-03-20T00:00:00.000000Z,1.833165e+09,PT.TIRTA BUANA MANDIRI
2,106,Aceh,2015,ocds-data-tender-2015-106.json,ocds-20h3g7-8605106,ocds-20h3g7-8605106-1,2015-03-23T00:00:00.000000Z,Arsip Nasional Republik Indonesia,L1,8605106,Pengadaan Jasa Kebersihan Rumah Sakit,services,7.890000e+08,2015-03-11T00:00:00.000000Z,complete,8605106-1,Pengadaan Jasa Kebersihan Rumah Sakit,2015-03-23T00:00:00.000000Z,7.429706e+08,PT.TIRTA BUANA MANDIRI


## 3. Standardize Column Names and Parse Types

In [3]:
data.columns = data.columns.str.strip().str.lower().str.replace(' ', '_')

date_cols = ['date', 'tender_datepublished', 'award_date']
for col in date_cols:
    data[col] = pd.to_datetime(data[col], utc=True, errors='coerce').dt.tz_localize(None)

num_cols = ['tender_minvalue', 'award_value']
for col in num_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

print('Columns after normalization:')
print(data.columns.tolist())
print('\nDtypes after parsing:')
print(data[date_cols + num_cols].dtypes.to_string())
print('\nMissing dates after parsing:')
print(data[date_cols].isnull().sum().to_string())


Columns after normalization:
['lspe_id', 'nama_daerah', 'source_year', 'source_file', 'ocid', 'release_id', 'date', 'buyer_name', 'buyer_id', 'tender_id', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished', 'tender_status', 'award_id', 'award_title', 'award_date', 'award_value', 'award_supplier']

Dtypes after parsing:
date                    datetime64[us]
tender_datepublished    datetime64[us]
award_date              datetime64[us]
tender_minvalue                float64
award_value                    float64

Missing dates after parsing:
date                    1
tender_datepublished    0
award_date              1


## 4. Remove Bad Rows

In [4]:
before = len(data)
data.drop_duplicates(inplace=True)
after_duplicates = len(data)

print(f'Duplicate rows removed: {before - after_duplicates:,}')

required_cols = ['lspe_id', 'nama_daerah', 'award_value', 'award_supplier', 'award_date', 'date']
before_required = len(data)
data.dropna(subset=required_cols, inplace=True)
after_required = len(data)

print(f'Rows removed for required nulls: {before_required - after_required:,}')

before_valid_dates = len(data)
data.dropna(subset=['date', 'award_date'], inplace=True)
after_valid_dates = len(data)

print(f'Rows removed for invalid parsed date/award_date: {before_valid_dates - after_valid_dates:,}')

before_positive = len(data)
data = data[data['award_value'] > 0].copy()
after_positive = len(data)

print(f'Rows removed for non-positive award_value: {before_positive - after_positive:,}')
print(f'Rows remaining after bad-row cleanup: {len(data):,}')


Duplicate rows removed: 0
Rows removed for required nulls: 1
Rows removed for invalid parsed date/award_date: 0
Rows removed for non-positive award_value: 0
Rows remaining after bad-row cleanup: 145,921


## 5. Basic Multi-Daerah Standardization

In [5]:
data['lspe_id'] = data['lspe_id'].astype(str).str.strip()
data['nama_daerah'] = data['nama_daerah'].apply(standardize_region_name)

print('Rows per daerah before imputation:')
print(data.groupby(['nama_daerah', 'lspe_id']).size().to_string())


Rows per daerah before imputation:
nama_daerah                 lspe_id
Aceh                        106        12475
Bali                        33          1934
Banten                      99          3452
Bengkulu                    267         1974
DKI Jakarta                 127         8678
Daerah Istimewa Yogyakarta  13          4537
Gorontalo                   18          1629
Jambi                       70          3000
Jawa Barat                  14         10629
Jawa Tengah                 42          6038
Jawa Timur                  15          6440
Kalimantan Barat            97          3015
Kalimantan Selatan          181         2460
Kalimantan Tengah           12          4120
Kalimantan Timur            35          4053
Kalimantan Utara            716         1897
Kepulauan Bangka Belitung   86          1342
Kepulauan Riau              22          2777
Lampung                     121         7083
Maluku                      288         5937
Maluku Utara                3

## 6. Recover Missing Buyer Name with Region Context

In [6]:
# buyer_name: try to recover from other rows sharing the same buyer_id within the same daerah
buyer_map = (
    data.dropna(subset=['buyer_name', 'buyer_id'])
        .drop_duplicates(['nama_daerah', 'buyer_id'])
        .set_index(['nama_daerah', 'buyer_id'])['buyer_name']
        .to_dict()
)

data['buyer_name'] = data['buyer_name'].fillna(
    data.apply(lambda row: buyer_map.get((row['nama_daerah'], row['buyer_id'])), axis=1)
)

# Remaining buyer_name nulls fill with region-specific province label
data['buyer_name'] = data['buyer_name'].fillna(
    'Pemerintah Daerah Provinsi ' + data['nama_daerah']
)

print('Missing buyer_name after recovery:')
print(int(data['buyer_name'].isnull().sum()))
print('\nSample buyer names:')
print(data['buyer_name'].value_counts().head(10).to_string())


Missing buyer_name after recovery:
0

Sample buyer names:
buyer_name
Pemerintah Daerah Provinsi Aceh                11270
Pemerintah Daerah Provinsi DKI Jakarta          8617
Pemerintah Daerah Provinsi Jawa Timur           6165
Pemerintah Daerah Provinsi Jawa Tengah          5894
Pemerintah Daerah Provinsi Sumatera Utara       5704
Pemerintah Daerah Provinsi Lampung              5390
Pemerintah Daerah Provinsi Maluku               5267
Pemerintah Daerah Provinsi Sumatera Selatan     5088
Pemerintah Daerah Provinsi Jawa Barat           4812
Pemerintah Daerah Provinsi Riau                 4626


## 7. Fill Remaining Missing Values

In [7]:
# mainprocurementcategory fill with 'Unknown Category'
data['mainprocurementcategory'] = data['mainprocurementcategory'].fillna('Unknown Category')

# tender_status fill with 'Unknown Status'
data['tender_status'] = data['tender_status'].fillna('Unknown Status')

# tender_title and award_title fill
data['tender_title'] = data['tender_title'].fillna('Unknown Tender Title')
data['award_title'] = data['award_title'].fillna(data['tender_title'])

# tender_minvalue fill missing with median per daerah first, then global median
median_min_by_region = data.groupby('nama_daerah')['tender_minvalue'].transform('median')
global_median_min = data['tender_minvalue'].median()
data['tender_minvalue'] = data['tender_minvalue'].fillna(median_min_by_region)
data['tender_minvalue'] = data['tender_minvalue'].fillna(global_median_min)

print('Missing values after general imputation:')
print(data[['mainprocurementcategory', 'tender_status', 'tender_title', 'award_title', 'tender_minvalue']].isnull().sum().to_string())


Missing values after general imputation:
mainprocurementcategory    0
tender_status              0
tender_title               0
award_title                0
tender_minvalue            0


## 8. Build Temporal and Ratio Features

In [8]:
data['days_to_award'] = (data['award_date'] - data['tender_datepublished']).dt.days

# fallback jika tender_datepublished kosong: gunakan selisih award_date dengan release date
fallback_days = (data['award_date'] - data['date']).dt.days
data['days_to_award'] = data['days_to_award'].fillna(fallback_days)
data['days_to_award'] = data['days_to_award'].fillna(0).clip(lower=0)

# budget_utilization_ratio: award_value dibanding tender_minvalue
ratio_denominator = data['tender_minvalue'].replace(0, np.nan)
data['budget_utilization_ratio'] = (
    data['award_value'] / ratio_denominator
).clip(lower=0, upper=10)

ratio_median_by_region = data.groupby('nama_daerah')['budget_utilization_ratio'].transform('median')
global_ratio_median = data['budget_utilization_ratio'].median()
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(ratio_median_by_region)
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(global_ratio_median)
data['budget_utilization_ratio'] = data['budget_utilization_ratio'].fillna(1.0)

fallback_days_count = int(data['tender_datepublished'].isnull().sum())

print('Temporal feature stats:')
print(data[['days_to_award']].describe().to_string())
print('\nRatio feature stats:')
print(data[['budget_utilization_ratio']].describe().to_string())
print(f'\nRows using fallback days_to_award logic: {fallback_days_count:,}')


Temporal feature stats:
       days_to_award
count  145921.000000
mean       28.866928
std       292.916788
min         0.000000
25%        14.000000
50%        20.000000
75%        32.000000
max     37435.000000

Ratio feature stats:
       budget_utilization_ratio
count             145921.000000
mean                   0.930498
std                    0.107580
min                    0.000011
25%                    0.899406
50%                    0.958524
75%                    0.984991
max                   10.000000

Rows using fallback days_to_award logic: 0


## 9. Clean Text Columns

In [9]:
text_cols = [
    'buyer_name', 'tender_title', 'mainprocurementcategory',
    'tender_status', 'award_title', 'award_supplier'
]

for col in text_cols:
    data[col] = (
        data[col].astype(str)
            .str.strip()
            .str.replace(r'\s+', ' ', regex=True)
            .str.title()
    )

print('Sample procurement categories:')
print(data['mainprocurementcategory'].value_counts().head(10).to_string())


Sample procurement categories:
mainprocurementcategory
Works       67497
Services    52455
Goods       25969


## 10. Drop Technical Identity Columns

In [10]:
cols_to_drop = ['ocid', 'release_id', 'buyer_id', 'award_id', 'tender_id']
data.drop(columns=[col for col in cols_to_drop if col in data.columns], inplace=True)

print('Remaining columns:')
print(data.columns.tolist())


Remaining columns:
['lspe_id', 'nama_daerah', 'source_year', 'source_file', 'date', 'buyer_name', 'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished', 'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier', 'days_to_award', 'budget_utilization_ratio']


## 11. Reorder Columns and Build Summary

In [11]:
column_order = [
    'lspe_id', 'nama_daerah', 'source_year', 'source_file', 'date', 'buyer_name',
    'tender_title', 'mainprocurementcategory', 'tender_minvalue', 'tender_datepublished',
    'tender_status', 'award_title', 'award_date', 'award_value', 'award_supplier',
    'days_to_award', 'budget_utilization_ratio'
]

remaining_columns = [col for col in data.columns if col not in column_order]
data = data[[col for col in column_order if col in data.columns] + remaining_columns]

summary_df = (
    data.groupby(['nama_daerah', 'lspe_id'])
        .agg(
            rows=('nama_daerah', 'size'),
            min_award_date=('award_date', 'min'),
            max_award_date=('award_date', 'max'),
            median_award_value=('award_value', 'median'),
            median_tender_minvalue=('tender_minvalue', 'median'),
            median_days_to_award=('days_to_award', 'median')
        )
        .reset_index()
        .sort_values(['nama_daerah', 'lspe_id'])
        .reset_index(drop=True)
)

print('Per-daerah cleaned summary:')
display(summary_df)


Per-daerah cleaned summary:


,nama_daerah,lspe_id,rows,min_award_date,max_award_date,median_award_value,median_tender_minvalue,median_days_to_award
0,Aceh,106,12475,2015-03-09,2023-10-25 00:00:00,5.576700e+08,5.992990e+08,25.0
1,Bali,33,1934,2014-12-10,2023-10-10 00:00:00,4.885572e+08,5.738740e+08,23.0
2,Banten,99,3452,2015-01-13,2023-11-02 00:00:00,4.547458e+08,4.981000e+08,22.0
3,Bengkulu,267,1974,2015-02-15,2023-11-06 00:00:00,6.482500e+08,6.605375e+08,15.0
4,DKI Jakarta,127,8678,2014-12-24,2023-10-18 00:00:00,7.585620e+08,8.799827e+08,21.0
5,Daerah Istimewa Yogyakarta,13,4537,2015-01-23,2024-10-24 14:00:00,3.806220e+08,4.393400e+08,17.0
6,Gorontalo,18,1629,2014-12-21,2024-09-13 15:30:00,3.962130e+08,4.610913e+08,26.0
7,Jambi,70,3000,2014-11-26,2023-09-30 00:00:00,6.803874e+08,7.000000e+08,27.0
8,Jawa Barat,14,10629,2014-12-15,2024-11-21 11:00:00,6.402672e+08,7.080600e+08,23.0
9,Jawa Tengah,42,6038,2014-11-29,2023-10-16 00:00:00,6.520898e+08,7.478834e+08,25.0


## 12. Save Output Files

In [12]:
data.to_csv(cleaned_data_path, index=False)
data.to_csv(normalized_data_path, index=False)
summary_df.to_csv(summary_path, index=False)

print('FINAL DATASET SUMMARY')
print('=' * 50)
print(f'Total rows   : {len(data):,}')
print(f'Total columns: {len(data.columns)}')
print('\nMissing values after cleaning:')
print(data.isnull().sum().to_string())
print('\nSaved files:')
print(f'- Cleaned dataset   : {cleaned_data_path.resolve()}')
print(f'- Normalized dataset: {normalized_data_path.resolve()}')
print(f'- Region summary    : {summary_path.resolve()}')

display(data.head(5))


FINAL DATASET SUMMARY
Total rows   : 145,921
Total columns: 17

Missing values after cleaning:
lspe_id                     0
nama_daerah                 0
source_year                 0
source_file                 0
date                        0
buyer_name                  0
tender_title                0
mainprocurementcategory     0
tender_minvalue             0
tender_datepublished        0
tender_status               0
award_title                 0
award_date                  0
award_value                 0
award_supplier              0
days_to_award               0
budget_utilization_ratio    0

Saved files:
- Cleaned dataset   : C:\Users\VICTUS\coding\collab\3\data\cleaned\cleaned_data.csv
- Normalized dataset: C:\Users\VICTUS\coding\collab\3\data\csv\normalized_data.csv
- Region summary    : C:\Users\VICTUS\coding\collab\3\data\cleaned\cleaned_data_summary_by_daerah.csv


,lspe_id,nama_daerah,source_year,source_file,date,buyer_name,tender_title,mainprocurementcategory,tender_minvalue,tender_datepublished,tender_status,award_title,award_date,award_value,award_supplier,days_to_award,budget_utilization_ratio
0,106,Aceh,2015,ocds-data-tender-2015-106.json,2015-03-09,Pemerintah Daerah Provinsi Aceh,Pembangunan Gedung Kantor Pt. Bank Aceh Cabang...,Works,1.335736e+10,2015-02-09,Complete,Pembangunan Gedung Kantor Pt. Bank Aceh Cabang...,2015-03-09,1.187118e+10,Pt Res Karya,28,0.888738
1,106,Aceh,2015,ocds-data-tender-2015-106.json,2015-03-20,Pemerintah Daerah Provinsi Aceh,Pengadaan Jasa Cleaning Servis,Services,2.291846e+09,2015-03-06,Complete,Pengadaan Jasa Cleaning Servis,2015-03-20,1.833165e+09,Pt.Tirta Buana Mandiri,14,0.799864
2,106,Aceh,2015,ocds-data-tender-2015-106.json,2015-03-23,Arsip Nasional Republik Indonesia,Pengadaan Jasa Kebersihan Rumah Sakit,Services,7.890000e+08,2015-03-11,Complete,Pengadaan Jasa Kebersihan Rumah Sakit,2015-03-23,7.429706e+08,Pt.Tirta Buana Mandiri,12,0.941661
3,106,Aceh,2015,ocds-data-tender-2015-106.json,2015-03-23,Arsip Nasional Republik Indonesia,Pengadaan Makanan Dan Minuman Pasien,Services,1.520736e+09,2015-03-11,Complete,Pengadaan Makanan Dan Minuman Pasien,2015-03-23,1.425690e+09,Cv. Berjaya,12,0.937500
4,106,Aceh,2015,ocds-data-tender-2015-106.json,2015-04-30,Pemerintah Daerah Provinsi Aceh,Pengawasan Pembangunan Landscape Dan Infrastru...,Services,7.166937e+09,2015-03-12,Complete,Pengawasan Pembangunan Landscape Dan Infrastru...,2015-04-30,6.421100e+09,Pt. Artefak Arkindo,49,0.895934
